# 04 — Champion Model: Logistic Scorecard

Traditional credit scorecard built on WoE-transformed features with
logistic regression. Converts log-odds to a points-based scale (300–850).

This is the **industry-standard** approach used by banks worldwide under
Basel II/III regulatory requirements.

In [ ]:
import sys
sys.path.insert(0, "../src")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

from woe_encoder import WoEEncoder
from evaluate import compute_gini, compute_ks_statistic

sns.set_style("whitegrid")
%matplotlib inline

In [ ]:
df = pd.read_csv("../data/processed/train_features.csv")
target = "TARGET"
exclude = ["SK_ID_CURR", "SK_ID_PREV", target]

numeric_cols = df.select_dtypes(include="number").columns.drop(exclude, errors="ignore").tolist()
X = df[numeric_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}  |  Default rate: {y.mean():.2%}")

## WoE Transform + Logistic Regression

In [ ]:
# Fit WoE on training data
encoder = WoEEncoder(bins=10)
X_train_woe = encoder.fit_transform(X_train, y_train).fillna(0)
X_test_woe = encoder.transform(X_test).fillna(0)

# Feature selection via IV
iv = encoder.get_iv_summary()
selected = iv[iv["IV"] >= 0.02].index.tolist()
print(f"Selected {len(selected)} features with IV >= 0.02")

X_train_sel = X_train_woe[selected]
X_test_sel = X_test_woe[selected]

In [ ]:
# Train logistic regression
lr = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    solver="lbfgs",
    random_state=42,
)
lr.fit(X_train_sel, y_train)

# Evaluate
y_prob = lr.predict_proba(X_test_sel)[:, 1]
auc = roc_auc_score(y_test, y_prob)
gini = compute_gini(y_test, y_prob)
ks = compute_ks_statistic(y_test, y_prob)

print(f"\nChampion Scorecard Performance:")
print(f"  AUC:  {auc:.4f}")
print(f"  Gini: {gini:.4f}")
print(f"  KS:   {ks:.4f}")

## Convert to Points-Based Scorecard

In [ ]:
# Scorecard parameters
PDO = 20          # Points to Double Odds
BASE_SCORE = 600  # Score at base odds
BASE_ODDS = 1/19  # ~5% default rate baseline

# Generate scores
log_odds = lr.decision_function(X_test_sel)
scores = np.array([WoEEncoder.log_odds_to_score(lo, pdo=PDO, base_score=BASE_SCORE, base_odds=BASE_ODDS)
                    for lo in log_odds])

# Score distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(scores[y_test == 0], bins=50, alpha=0.6, label="Good (non-default)", density=True)
ax.hist(scores[y_test == 1], bins=50, alpha=0.6, label="Bad (default)", density=True)
ax.set_xlabel("Credit Score")
ax.set_ylabel("Density")
ax.set_title("Score Distribution — Champion Scorecard")
ax.legend()
plt.tight_layout()
plt.show()

## Feature Importance (Scorecard Coefficients)

In [ ]:
coef_df = (
    pd.DataFrame({"Feature": selected, "Coefficient": lr.coef_[0]})
    .sort_values("Coefficient", key=abs, ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = coef_df["Coefficient"].apply(lambda x: "coral" if x > 0 else "steelblue")
ax.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
ax.set_xlabel("Logistic Regression Coefficient")
ax.set_title("Top 20 Scorecard Features (WoE-space)")
plt.tight_layout()
plt.show()

## Calibration Plot

A well-calibrated model means predicted probabilities match observed default rates. For credit risk, calibration is critical — the predicted PD directly feeds into pricing, provisioning, and capital calculations. A model with good AUC but poor calibration will systematically over- or under-price risk.

In [ ]:
# Calibration curve for the champion scorecard
from sklearn.calibration import calibration_curve

y_prob_champion = lr.predict_proba(X_test_sel)[:, 1]
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, y_prob_champion, n_bins=10, strategy='uniform'
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Reliability diagram
ax1.plot(mean_predicted_value, fraction_of_positives, 's-', label='Scorecard', color='#2c3e50')
ax1.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated', alpha=0.5)
ax1.set_xlabel('Mean Predicted Probability')
ax1.set_ylabel('Observed Default Rate')
ax1.set_title('Calibration Curve (Reliability Diagram)')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Prediction distribution
ax2.hist(y_prob_champion, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
ax2.set_xlabel('Predicted Probability of Default')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Predicted Probabilities')
ax2.axvline(x=y_test.mean(), color='red', linestyle='--', label=f'Base rate: {y_test.mean():.3f}')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Brier score (lower = better calibrated)
from sklearn.metrics import brier_score_loss
brier = brier_score_loss(y_test, y_prob_champion)
print(f"Brier Score: {brier:.4f} (lower = better calibration)")
print(f"Expected Calibration Error (ECE): {np.mean(np.abs(fraction_of_positives - mean_predicted_value)):.4f}")

In [ ]:
# Save artefacts
import pathlib
models_dir = pathlib.Path("../models")
models_dir.mkdir(exist_ok=True)

joblib.dump(lr, models_dir / "scorecard_champion.pkl")
joblib.dump(encoder, models_dir / "woe_encoder.pkl")
print("Champion model and encoder saved.")